In [1]:
pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.3 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

In [3]:
(x_train, y_train), (x_val, y_val) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255
x_val = x_val.reshape(-1, 784).astype("float32") / 255


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(
            keras.layers.Dense(
                units=hp.Int('units_' + str(i), min_value=32, max_value=256, step=32),
                activation='relu'
            )
        )
    model.add(keras.layers.Dense(10, activation='softmax'))
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    model.compile(optimizer=hp_optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [5]:
tuner = kt.RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials=5,
    executions_per_trial=2,
    directory='keras_tuner_demo',
    project_name='mnist_tuning'

)

In [6]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))

Trial 5 Complete [00h 01m 23s]
val_accuracy: 0.9553500115871429

Best val_accuracy So Far: 0.9796999990940094
Total elapsed time: 00h 08m 19s


In [8]:
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [9]:
best_model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))
test_loss, test_acc = best_model.evaluate(x_val, y_val)
print("Final Test Accuracy:", test_acc)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9904 - loss: 0.0289 - val_accuracy: 0.9801 - val_loss: 0.0690
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9936 - loss: 0.0211 - val_accuracy: 0.9791 - val_loss: 0.0704
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9942 - loss: 0.0178 - val_accuracy: 0.9794 - val_loss: 0.0742
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9955 - loss: 0.0142 - val_accuracy: 0.9788 - val_loss: 0.0802
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9960 - loss: 0.0123 - val_accuracy: 0.9785 - val_loss: 0.0857
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9967 - loss: 0.0099 - val_accuracy: 0.9779 - val_loss: 0.0879
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9974 - loss: 0.0082 - val_accuracy: 0.9771 - val_loss: 0.0914
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9968 - loss: 0.